# Start here: Iteration 2

This notebook is the entry point. **Run All is safe: it only inspects local file availability.** It does not convert data, fit models, or run reserved-test evaluation.

1. Select the Iteration2 `.venv` kernel (see [setup and full guide](README.md)).
2. Run the check below to see whether your five-turbine/five-year dataset is available.
3. Open the appropriate linked stage below. You do not need to run every notebook.


In [ ]:
from pathlib import Path

ROOT = next((candidate for parent in (Path.cwd(), *Path.cwd().parents)
             for candidate in (parent, parent / 'iteration2' / 'Iteration2')
             if (candidate / 'notebooks' / '03_preprocessing').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open the Iteration2 workspace or run from within this project.')
YEARS = [2016, 2017, 2018, 2020, 2021]
TURBINE_IDS = [str(number) for number in range(2304510, 2304515)]
ready, missing, pointers, invalid = [], [], [], []
for year in YEARS:
    for turbine in TURBINE_IDS:
        path = ROOT / 'Data_Parquets_10min' / str(year) / f'Station_{turbine}' / 'data.parquet'
        if not path.is_file():
            missing.append(str(path.relative_to(ROOT)))
            continue
        with path.open('rb') as stream:
            header = stream.read(128)
        if header.startswith(b'version https://git-lfs.github.com/spec/v1'):
            pointers.append(str(path.relative_to(ROOT)))
        elif header[:4] != b'PAR1':
            invalid.append(str(path.relative_to(ROOT)))
        else:
            ready.append(path)
print(f'Project: {ROOT}')
print(f'Processed files ready: {len(ready)}/25 (header check only, not a schema validation).')
if len(ready) == 25:
    print('Start at Stage 4 (optional plots) or Stage 5 (one training experiment). Skip conversion.')
else:
    print('Missing files:', len(missing), '| LFS pointers:', len(pointers), '| unexpected headers:', len(invalid))
    if pointers:
        print('Run git lfs install and git lfs pull from the repository, then rerun this check.')
    for name in (missing + pointers + invalid)[:10]:
        print(' -', name)
    workbooks = list((ROOT / 'Data' / 'Interim' / 'turbine_exports').glob('turbines_*.xlsx'))
    sources = list((ROOT / 'Data_Parquets').glob('*/Station_*/sensors.parquet'))
    print(f'Available yearly workbooks: {len(workbooks)}; source sensor Parquets: {len(sources)}.')
    print('Use Stage 3 for existing source Parquets, Stage 2 for Excel, or Stage 1 for raw CSVs.')
print('Models: constant, random-weight, lightweight logistic regression, random forest, LightGBM, XGBoost.')
print('Stage 5 split: train 2016-2018; validate 2020; test 2021. Previous two hours -> selected future horizon.')


## Open the next notebook

| Stage | Notebook / guide |
|---|---|
| 1: Raw CSV → Excel | [Export yearly data](notebooks/01_data_export/01_export_yearly_turbine_scada.ipynb) |
| 2: Excel → source Parquets | [Set up yearly Parquets](notebooks/02_parquet_setup/01_setup_yearly_parquets.ipynb) |
| 3: Source → ten-minute Parquets | [Preprocess](notebooks/03_preprocessing/01_preprocess_10min_parquets.ipynb) |
| 4: Inspect | [Table preview](notebooks/02_parquet_setup/preview_parquet.ipynb) / [plot choices](notebooks/04_data_visualisation/README.md) |
| 5: Recommended first experiment | [All alarms, 10 minutes, all features](notebooks/05_model_training/all_alarms/01_horizon_10min/01_train_all_features.ipynb) |
| 5: Other targets / horizons | [Model training directory](notebooks/05_model_training/README.md) |
| 6: Advanced alternative | [All-turbine temporal CV](notebooks/06_all_turbine_cross_validation/README.md) |
| 7: Read completed CV results | [CV evaluation](evaluate_cross_validation_results.ipynb) |

**Stage 5 experiments are independent:** select one target, one horizon and one feature mode. `02` and `03` within a horizon are alternative feature configurations, not mandatory next steps. All-alarm prediction includes operational codes 20 and 25; choose an alarm-specific group for a narrower target.

**Stage 6 has different settings:** all eligible turbines, development 2016-2023, six-hour horizon, reserved testing 2024-2025 off by default. It does not require Stage 5 models. Stage 7 only reads completed reports; confirm the saved target and configuration before interpreting historical results.

After training, use the output path printed by the notebook. Review metrics, PR curves, confusion matrices and `configuration.json` there. **Restart Kernel → Run All** whenever changing years, turbines, features or target settings.
